<a href="https://colab.research.google.com/github/PinkUnicorm/CNNs-To-Detect-Cancer-Anomaly-s-in-Patients-CT-Scans/blob/main/Copy_of_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1 — Download and import the dataset in Google Colab

!pip -q install kagglehub pandas

from pathlib import Path
import pandas as pd
import kagglehub


kagglehub.login()

# Download the dataset
dataset_path = Path(kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset"))
print("Dataset downloaded to:", dataset_path)

# Exclude *_mask.png files since those are segmentation masks
classes = {"benign", "malignant", "normal"}
records = []

for img_path in dataset_path.rglob("*.png"):
    if "_mask" in img_path.stem.lower():
        continue

    parts_lower = [part.lower() for part in img_path.parts]
    label = next((c for c in classes if c in parts_lower), None)

    # Fallback in case class name is embedded in filename
    if label is None:
        fname = img_path.stem.lower()
        label = next((c for c in classes if c in fname), None)

    if label is not None:
        records.append({
            "image_path": str(img_path),
            "label": label
        })

df = pd.DataFrame(records)

print("\nFirst 5 rows:")
print(df.head())

print("\nClass distribution:")
print(df["label"].value_counts())

print(f"\nTotal usable images: {len(df)}")

Using Colab cache for faster access to the 'breast-ultrasound-images-dataset' dataset.
Dataset downloaded to: /kaggle/input/breast-ultrasound-images-dataset


In [ ]:
# Step 2 — Sanity check the dataset and visualize samples

import matplotlib.pyplot as plt
from PIL import Image
import random

print("Number of images:", len(df))
print("\nClass counts:")
print(df["label"].value_counts())

# Check image sizes
size_records = []
for path in df["image_path"].sample(min(100, len(df)), random_state=42):
    try:
        with Image.open(path) as img:
            size_records.append(img.size)   # (width, height)
    except Exception as e:
        print("Could not open:", path, e)

print("\nSample image sizes (first 10):", size_records[:10])

# Show a few example images from each class
classes = sorted(df["label"].unique())
samples_per_class = 3

fig, axes = plt.subplots(len(classes), samples_per_class, figsize=(12, 10))

for i, label in enumerate(classes):
    class_df = df[df["label"] == label]
    sample_rows = class_df.sample(min(samples_per_class, len(class_df)), random_state=42)

    for j, (_, row) in enumerate(sample_rows.iterrows()):
        img = Image.open(row["image_path"]).convert("L")  # ultrasound images are effectively grayscale
        axes[i, j].imshow(img, cmap="gray")
        axes[i, j].set_title(f"{label}")
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Step 3 — Train / validation / test split with stratification

from sklearn.model_selection import train_test_split

# First split: train vs temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,          # 70% train, 30% temp
    stratify=df["label"],
    random_state=42
)

# Second split: validation vs test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,          # 15% validation, 15% test
    stratify=temp_df["label"],
    random_state=42
)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Show split sizes
print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

print("\nTrain class distribution:")
print(train_df["label"].value_counts())

print("\nValidation class distribution:")
print(val_df["label"].value_counts())

print("\nTest class distribution:")
print(test_df["label"].value_counts())

In [ ]:
#Step 4 Encode the class labels
#Convert string labels like benign, malignant, and normal into numeric form for model training.

from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Fit label encoder on all labels
label_encoder = LabelEncoder()
label_encoder.fit(df["label"])

# Transform labels for each split
train_df["label_encoded"] = label_encoder.transform(train_df["label"])
val_df["label_encoded"] = label_encoder.transform(val_df["label"])
test_df["label_encoded"] = label_encoder.transform(test_df["label"])

# Optional: one-hot encoded labels for Keras multiclass classification
num_classes = len(label_encoder.classes_)

y_train = to_categorical(train_df["label_encoded"], num_classes=num_classes)
y_val = to_categorical(val_df["label_encoded"], num_classes=num_classes)
y_test = to_categorical(test_df["label_encoded"], num_classes=num_classes)

# Show mapping
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Label mapping:", label_mapping)

print("\nEncoded train labels preview:")
print(train_df[["label", "label_encoded"]].head())

print("\nNumber of classes:", num_classes)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

In [ ]:
#Step 5 Preprocess the images Resize all images to a fixed size, normalize pixel values, and convert them into the format expected by the models.
import tensorflow as tf

# Image / training settings
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess_image(path, label):
    # Read image
    img = tf.io.read_file(path)

    # Decode image (force 3 channels so it works with CNNs and pretrained models)
    img = tf.image.decode_png(img, channels=3)

    # Resize to fixed size
    img = tf.image.resize(img, IMG_SIZE)

    # Normalize pixel values to [0, 1]
    img = tf.cast(img, tf.float32) / 255.0

    return img, label

def make_dataset(dataframe, shuffle=False):
    paths = dataframe["image_path"].values
    labels = dataframe["label_encoded"].values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), reshuffle_each_iteration=True)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

# Build datasets
train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
test_ds = make_dataset(test_df, shuffle=False)

# Check one batch
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Pixel range:", tf.reduce_min(images).numpy(), "to", tf.reduce_max(images).numpy())


In [ ]:
#Step 6 Apply data augmentation to the training set Add transformations such as rotation, zoom, flipping, shifting, or brightness changes to improve

from tensorflow.keras import layers

# Mild augmentation pipeline for medical images
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.05),        # small rotation
    layers.RandomZoom(0.10),            # small zoom in/out
    layers.RandomTranslation(0.05, 0.05),  # slight shifts
    layers.RandomContrast(0.10)         # slight contrast variation
], name="data_augmentation")

def augment(image, label):
    image = data_augmentation(image, training=True)
    return image, label

# Apply augmentation only to the training dataset
augmented_train_ds = train_ds.map(augment, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# Quick check
for images, labels in augmented_train_ds.take(1):
    print("Augmented image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Pixel range:", tf.reduce_min(images).numpy(), "to", tf.reduce_max(images).numpy())

In [ ]:
#Step 7 Create training, validation, and test data pipelines Build TensorFlow or PyTorch data loaders/generators so batches can be fed efficiently into the models.

from tensorflow.keras import layers, models

cnn_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    # Convolution block 1
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Convolution block 2
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Convolution block 3
    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Convolution block 4
    layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Classifier
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])

cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

In [ ]:
#Step 8 Build the baseline CNN model Create a custom CNN for the classification task as your first main approach. This is one of the two core algorithms in your project.

print("Classes:", list(label_encoder.classes_))
print("Number of classes:", num_classes)
print()
cnn_model.summary()

In [ ]:
#Step 9 Train the baseline CNN Fit the CNN on the training set while monitoring validation performance across epochs.

import os
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt

# callbacks

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath="best_cnn_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

# train
EPOCHS = 30

history = cnn_model.fit(
    augmented_train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

# plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# accuracy
axes[0].plot(history.history["accuracy"], label="Train Accuracy", linewidth=2)
axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy", linewidth=2, linestyle="--")
axes[0].set_title("CNN - Accuracy over Epochs", fontsize=13)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

#Loss
axes[1].plot(history.history["loss"], label="Train Loss", linewidth=2)
axes[1].plot(history.history["val_loss"], label="Validation Loss", linewidth=2, linestyle="--")
axes[1].set_title("CNN - Loss over Epochs", fontsize=13)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Baseline CNN Training History", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("cnn_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Training complete. Best model saved to: best_cnn_model.keras")

In [ ]:
#Step 10 Evaluate the baseline CNN Measure accuracy, precision, recall, F1 score, and confusion matrix, since these are the metrics listed in your project.

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# get true labels and predictions from test set
y_true = []
y_pred_probs = []

for images, labels in test_ds:
  probs = cnn_model.predict(images, verbose=0)
  y_pred_probs.extend(probs)
  y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = np.argmax(y_pred_probs, axis=1) # integer class predictions

class_names = list(label_encoder.classes_) # ['benign', 'malignant', 'normal']

# scalar metrics
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true, y_pred, average="weighted",zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("=" * 45)
print("       Baseline CNN — Test Set Results")
print("=" * 45)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision : {precision:.4f}  (weighted)")
print(f"  Recall    : {recall:.4f}  (weighted)")
print(f"  F1 Score  : {f1:.4f}  (weighted)")
print("=" * 45)

# per-class report
print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

# confusion matrix
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# raw counts
sns.heatmap(
    cm,
    annot=True, fmt='d',
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues",
    linewidth=0.5,
    ax=axes[0]
)

# normalized (row %)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
sns.heatmap(
    cm_norm,
    annot=True, fmt=".2f",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues",
    vmin=0, vmax=1,
    linewidths=0.5,
    ax=axes[1]
)
axes[1].set_title("Confusion Matrix (row-normalized)", fontsize=13)
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Label")

plt.suptitle("Baseline CNN — Confusion Matrices", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("cnn_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#Step 11 Build the transfer learning model

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_SIZE = (224, 224)
INPUT_SHAPE = (224, 224, 3)

# Apply the correct preprocessing for MobileNetV2
def apply_mobilenet_preprocessing(image, label):
    image = preprocess_input(image * 255.0)
    # because your images are currently scaled to [0,1], MobileNetV2 expects pixels in [-1,1]
    return image, label

# Build datasets for transfer learning
train_tl_ds = augmented_train_ds.map(apply_mobilenet_preprocessing, num_parallel_calls=tf.data.AUTOTUNE)
val_tl_ds = val_ds.map(apply_mobilenet_preprocessing, num_parallel_calls=tf.data.AUTOTUNE)
test_tl_ds = test_ds.map(apply_mobilenet_preprocessing, num_parallel_calls=tf.data.AUTOTUNE)

# Load pretrained base model
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=INPUT_SHAPE
)

# Freeze the base model for initial training
base_model.trainable = False

# Build the full model
transfer_model = models.Sequential([
    layers.Input(shape=INPUT_SHAPE),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Transfer learning model summary:")
transfer_model.summary()


In [ ]:
#Step 12 Train the transfer learning model
#Step 12 Train the transfer learning model

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt

# Callbacks
early_stop_tl = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

checkpoint_tl = ModelCheckpoint(
    filepath="best_transfer_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

# Phase 1: Train classifier head
initial_epochs = 10

history_tl_1 = transfer_model.fit(
    train_tl_ds,
    validation_data=val_tl_ds,
    epochs=initial_epochs,
    callbacks=[early_stop_tl, checkpoint_tl],
    verbose=1
)

# Phase 2: Fine-tune top layers
base_model.trainable = True

# Freeze most layers, unfreeze only top part
fine_tune_at = 100  # unfreeze from this layer onward

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fine_tune_epochs = 10
total_epochs = initial_epochs + fine_tune_epochs

history_tl_2 = transfer_model.fit(
    train_tl_ds,
    validation_data=val_tl_ds,
    epochs=total_epochs,
    initial_epoch=history_tl_1.epoch[-1] + 1,
    callbacks=[early_stop_tl, checkpoint_tl],
    verbose=1
)

# Combine history for plotting
acc = history_tl_1.history["accuracy"] + history_tl_2.history["accuracy"]
val_acc = history_tl_1.history["val_accuracy"] + history_tl_2.history["val_accuracy"]
loss = history_tl_1.history["loss"] + history_tl_2.history["loss"]
val_loss = history_tl_1.history["val_loss"] + history_tl_2.history["val_loss"]

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(acc, label="Train Accuracy", linewidth=2)
axes[0].plot(val_acc, label="Validation Accuracy", linewidth=2, linestyle="--")
axes[0].axvline(x=initial_epochs-1, color="gray", linestyle=":", label="Start Fine-Tuning")
axes[0].set_title("Transfer Learning - Accuracy over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss, label="Train Loss", linewidth=2)
axes[1].plot(val_loss, label="Validation Loss", linewidth=2, linestyle="--")
axes[1].axvline(x=initial_epochs-1, color="gray", linestyle=":", label="Start Fine-Tuning")
axes[1].set_title("Transfer Learning - Loss over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Transfer Learning Training History", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("transfer_learning_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print("Training complete. Best model saved to: best_transfer_model.keras")

In [ ]:
#Step 13 Evaluate the transfer learning model

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Get predictions
y_true_tl = []
y_pred_probs_tl = []

for images, labels in test_tl_ds:
    probs = transfer_model.predict(images, verbose=0)
    y_pred_probs_tl.extend(probs)
    y_true_tl.extend(labels.numpy())

y_true_tl = np.array(y_true_tl)
y_pred_probs_tl = np.array(y_pred_probs_tl)
y_pred_tl = np.argmax(y_pred_probs_tl, axis=1)

class_names = list(label_encoder.classes_)

# Scalar metrics
acc_tl = accuracy_score(y_true_tl, y_pred_tl)
precision_tl = precision_score(y_true_tl, y_pred_tl, average="weighted", zero_division=0)
recall_tl = recall_score(y_true_tl, y_pred_tl, average="weighted", zero_division=0)
f1_tl = f1_score(y_true_tl, y_pred_tl, average="weighted", zero_division=0)

print("=" * 50)
print("   Transfer Learning Model — Test Set Results")
print("=" * 50)
print(f"  Accuracy  : {acc_tl:.4f}  ({acc_tl*100:.2f}%)")
print(f"  Precision : {precision_tl:.4f}  (weighted)")
print(f"  Recall    : {recall_tl:.4f}  (weighted)")
print(f"  F1 Score  : {f1_tl:.4f}  (weighted)")
print("=" * 50)

print("\nDetailed Classification Report:")
print(classification_report(y_true_tl, y_pred_tl, target_names=class_names, zero_division=0))

# Confusion matrix
cm_tl = confusion_matrix(y_true_tl, y_pred_tl)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    cm_tl,
    annot=True, fmt="d",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues",
    linewidths=0.5,
    ax=axes[0]
)
axes[0].set_title("Confusion Matrix (Counts)")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

cm_tl_norm = cm_tl.astype("float") / cm_tl.sum(axis=1, keepdims=True)
sns.heatmap(
    cm_tl_norm,
    annot=True, fmt=".2f",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues",
    vmin=0, vmax=1,
    linewidths=0.5,
    ax=axes[1]
)
axes[1].set_title("Confusion Matrix (Row-Normalized)")
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Label")

plt.suptitle("Transfer Learning Model — Confusion Matrices", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("transfer_learning_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
#Step 14 Compare both models Compare CNN vs transfer learning using validation/test metrics, confusion matrices


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Recompute metrics directly from predictions so this cell is independent of any overwritten variable names
cnn_metrics = {
    "Model": "Baseline CNN",
    "Test Accuracy": accuracy_score(y_true, y_pred),
    "Test Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
    "Test Recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
    "Test F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    "Best Val Accuracy": max(history.history.get("val_accuracy", [np.nan]))
}

tl_val_acc_history = history_tl_1.history.get("val_accuracy", []) + history_tl_2.history.get("val_accuracy", [])
tl_metrics = {
    "Model": "Transfer Learning (MobileNetV2)",
    "Test Accuracy": accuracy_score(y_true_tl, y_pred_tl),
    "Test Precision": precision_score(y_true_tl, y_pred_tl, average="weighted", zero_division=0),
    "Test Recall": recall_score(y_true_tl, y_pred_tl, average="weighted", zero_division=0),
    "Test F1": f1_score(y_true_tl, y_pred_tl, average="weighted", zero_division=0),
    "Best Val Accuracy": max(tl_val_acc_history) if len(tl_val_acc_history) > 0 else np.nan
}

comparison_df = pd.DataFrame([cnn_metrics, tl_metrics]).set_index("Model")
comparison_df_rounded = comparison_df.round(4)

print("=" * 70)
print("Model Comparison Summary")
print("=" * 70)
display(comparison_df_rounded)

# Highlight the better model for each metric
better_model = comparison_df.idxmax(axis=0).to_frame(name="Better Model")
print("\nBetter model by metric:")
display(better_model)

# Bar chart comparing test metrics
metric_names = ["Test Accuracy", "Test Precision", "Test Recall", "Test F1", "Best Val Accuracy"]
x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width/2, comparison_df.loc["Baseline CNN", metric_names], width, label="Baseline CNN")
ax.bar(x + width/2, comparison_df.loc["Transfer Learning (MobileNetV2)", metric_names], width, label="Transfer Learning")

ax.set_title("Baseline CNN vs Transfer Learning", fontsize=15, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(metric_names, rotation=20, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("model_comparison_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

# Compare validation accuracy histories
cnn_val_acc = history.history.get("val_accuracy", [])
tl_val_acc = tl_val_acc_history

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(cnn_val_acc) + 1), cnn_val_acc, marker="o", label="Baseline CNN")
plt.plot(range(1, len(tl_val_acc) + 1), tl_val_acc, marker="o", label="Transfer Learning")
plt.title("Validation Accuracy Across Epochs", fontsize=14, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("validation_accuracy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

winner = comparison_df["Test F1"].idxmax()
print(f"Overall, the stronger model on the test set is: {winner}")

In [ ]:
#Step 15 Perform error analysis Look at misclassified images, identify patterns in failures
# Inspect misclassified test images and identify the most common failure patterns.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Build result tables aligned with the original order of test_df
results_cnn_df = test_df.copy().reset_index(drop=True)
results_cnn_df["true_encoded"] = y_true
results_cnn_df["pred_encoded"] = y_pred
results_cnn_df["true_label"] = label_encoder.inverse_transform(results_cnn_df["true_encoded"])
results_cnn_df["pred_label"] = label_encoder.inverse_transform(results_cnn_df["pred_encoded"])
results_cnn_df["confidence"] = np.max(y_pred_probs, axis=1)
results_cnn_df["correct"] = results_cnn_df["true_encoded"] == results_cnn_df["pred_encoded"]

results_tl_df = test_df.copy().reset_index(drop=True)
results_tl_df["true_encoded"] = y_true_tl
results_tl_df["pred_encoded"] = y_pred_tl
results_tl_df["true_label"] = label_encoder.inverse_transform(results_tl_df["true_encoded"])
results_tl_df["pred_label"] = label_encoder.inverse_transform(results_tl_df["pred_encoded"])
results_tl_df["confidence"] = np.max(y_pred_probs_tl, axis=1)
results_tl_df["correct"] = results_tl_df["true_encoded"] == results_tl_df["pred_encoded"]

def summarize_errors(results_df, model_name):
    total_errors = int((~results_df["correct"]).sum())
    total_samples = len(results_df)
    print("=" * 70)
    print(f"{model_name} — Error Analysis")
    print("=" * 70)
    print(f"Misclassified images: {total_errors} / {total_samples} ({100 * total_errors / total_samples:.2f}%)")

    errors_only = results_df[~results_df["correct"]].copy()

    if len(errors_only) == 0:
        print("No misclassifications found.")
        return errors_only

    print("\nErrors by true class:")
    display(errors_only["true_label"].value_counts().rename("count").to_frame())

    print("\nMost common confusion pairs:")
    confusion_pairs = (
        errors_only.groupby(["true_label", "pred_label"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    display(confusion_pairs.head(10))

    return errors_only.sort_values("confidence", ascending=False).reset_index(drop=True)

cnn_errors = summarize_errors(results_cnn_df, "Baseline CNN")
tl_errors = summarize_errors(results_tl_df, "Transfer Learning (MobileNetV2)")

def show_misclassified_examples(errors_df, model_name, num_examples=6):
    if len(errors_df) == 0:
        print(f"{model_name}: no misclassified images to display.")
        return

    samples = errors_df.head(min(num_examples, len(errors_df))).copy()

    cols = 3
    rows = int(np.ceil(len(samples) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(14, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, samples.iterrows()):
        img = Image.open(row["image_path"]).convert("RGB").resize(IMG_SIZE)
        ax.imshow(img)
        ax.set_title(
            f"True: {row['true_label']}\nPred: {row['pred_label']}\nConf: {row['confidence']:.3f}",
            fontsize=10
        )
        ax.axis("off")

    for ax in axes[len(samples):]:
        ax.axis("off")

    plt.suptitle(f"{model_name} — Most Confident Misclassifications", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()

show_misclassified_examples(cnn_errors, "Baseline CNN")
show_misclassified_examples(tl_errors, "Transfer Learning (MobileNetV2)")

# Side-by-side class error rates
error_rate_by_class = pd.DataFrame({
    "Baseline CNN Error Rate": 1 - results_cnn_df.groupby("true_label")["correct"].mean(),
    "Transfer Learning Error Rate": 1 - results_tl_df.groupby("true_label")["correct"].mean()
}).sort_index()

print("\nPer-class error rates:")
display(error_rate_by_class.round(4))

ax = error_rate_by_class.plot(kind="bar", figsize=(10, 5))
ax.set_title("Per-Class Error Rate Comparison", fontsize=14, fontweight="bold")
ax.set_xlabel("True Class")
ax.set_ylabel("Error Rate")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("error_rate_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Step 16 — Generate explainability visualizations (Grad-CAM)
# Fixed version: avoids using transfer_model.layers[0].output directly

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib import cm
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Conv2D, DepthwiseConv2D, SeparableConv2D

# ------------------------------------------------------------
# 1. Make sure the transfer model is built/called
# ------------------------------------------------------------
if "INPUT_SHAPE" not in globals():
    INPUT_SHAPE = (224, 224, 3)

if "IMG_SIZE" not in globals():
    IMG_SIZE = (224, 224)

_ = transfer_model(tf.zeros((1, *INPUT_SHAPE), dtype=tf.float32), training=False)

# ------------------------------------------------------------
# 2. Find the MobileNetV2 base model inside transfer_model
# ------------------------------------------------------------
base_model_for_gradcam = None

for layer in transfer_model.layers:
    if isinstance(layer, tf.keras.Model) and "mobilenet" in layer.name.lower():
        base_model_for_gradcam = layer
        break

if base_model_for_gradcam is None:
    for layer in transfer_model.layers:
        if isinstance(layer, tf.keras.Model):
            base_model_for_gradcam = layer
            break

if base_model_for_gradcam is None:
    raise ValueError("Could not find the pretrained base model inside transfer_model.")

# ------------------------------------------------------------
# 3. Find the last convolutional layer in the base model
# ------------------------------------------------------------
last_conv_layer = None
for layer in reversed(base_model_for_gradcam.layers):
    if isinstance(layer, (Conv2D, DepthwiseConv2D, SeparableConv2D)):
        last_conv_layer = layer
        break

if last_conv_layer is None:
    raise ValueError("Could not find a convolutional layer in the base model.")

print("Using base model:", base_model_for_gradcam.name)
print("Using last conv layer:", last_conv_layer.name)

# ------------------------------------------------------------
# 4. Build:
#    - feature extractor: input image -> last conv feature map
#    - classifier head: feature map -> final prediction
# ------------------------------------------------------------
feature_extractor = tf.keras.Model(
    inputs=base_model_for_gradcam.input,
    outputs=last_conv_layer.output
)

classifier_input = tf.keras.Input(shape=last_conv_layer.output.shape[1:])
x = classifier_input

passed_base_model = False
for layer in transfer_model.layers:
    if layer == base_model_for_gradcam:
        passed_base_model = True
        continue
    if passed_base_model:
        x = layer(x)

classifier_head = tf.keras.Model(classifier_input, x)

# ------------------------------------------------------------
# 5. Helper functions
# ------------------------------------------------------------
def load_image_for_gradcam(img_path, target_size=IMG_SIZE):
    img = Image.open(img_path).convert("RGB").resize(target_size)
    img_array = np.array(img).astype("float32") / 255.0

    batch = np.expand_dims(img_array, axis=0)
    batch_preprocessed = preprocess_input(batch * 255.0)  # MobileNetV2 preprocessing

    return img_array, batch_preprocessed

def make_gradcam_heatmap(batch_preprocessed, feature_extractor, classifier_head, pred_index=None):
    batch_tensor = tf.convert_to_tensor(batch_preprocessed, dtype=tf.float32)

    with tf.GradientTape() as tape:
        conv_outputs = feature_extractor(batch_tensor, training=False)
        tape.watch(conv_outputs)

        predictions = classifier_head(conv_outputs, training=False)

        if pred_index is None:
            pred_index = tf.argmax(predictions[0])

        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)

    max_val = tf.reduce_max(heatmap)
    if max_val > 0:
        heatmap = heatmap / max_val

    return heatmap.numpy(), int(pred_index.numpy()), predictions.numpy()[0]

def overlay_heatmap(img_array, heatmap, alpha=0.35):
    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_img = Image.fromarray(heatmap_uint8).resize((img_array.shape[1], img_array.shape[0]))
    heatmap_resized = np.array(heatmap_img) / 255.0

    colored_heatmap = cm.get_cmap("jet")(heatmap_resized)[..., :3]
    overlay = np.clip((1 - alpha) * img_array + alpha * colored_heatmap, 0, 1)

    return overlay

# ------------------------------------------------------------
# 6. Rebuild results_tl_df if Step 15 was not run
# ------------------------------------------------------------
if "results_tl_df" not in globals():
    results_tl_df = test_df.copy().reset_index(drop=True)
    results_tl_df["true_encoded"] = y_true_tl
    results_tl_df["pred_encoded"] = y_pred_tl
    results_tl_df["true_label"] = label_encoder.inverse_transform(results_tl_df["true_encoded"])
    results_tl_df["pred_label"] = label_encoder.inverse_transform(results_tl_df["pred_encoded"])
    results_tl_df["confidence"] = np.max(y_pred_probs_tl, axis=1)
    results_tl_df["correct"] = results_tl_df["true_encoded"] == results_tl_df["pred_encoded"]

# ------------------------------------------------------------
# 7. Choose examples for Grad-CAM
#    First a few errors, then a few correct predictions
# ------------------------------------------------------------
error_examples = results_tl_df[~results_tl_df["correct"]].head(3)
correct_examples = results_tl_df[results_tl_df["correct"]].head(3)
gradcam_examples = pd.concat([error_examples, correct_examples], axis=0).reset_index(drop=True)

# fallback if no rows found for some reason
if len(gradcam_examples) == 0:
    gradcam_examples = results_tl_df.head(6).reset_index(drop=True)

# ------------------------------------------------------------
# 8. Generate and display Grad-CAM results
# ------------------------------------------------------------
if len(gradcam_examples) == 0:
    print("No examples available for Grad-CAM.")
else:
    fig, axes = plt.subplots(len(gradcam_examples), 3, figsize=(12, 4 * len(gradcam_examples)))

    if len(gradcam_examples) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, row in gradcam_examples.iterrows():
        img_array, batch_preprocessed = load_image_for_gradcam(row["image_path"])
        heatmap, pred_idx, pred_probs = make_gradcam_heatmap(
            batch_preprocessed,
            feature_extractor,
            classifier_head
        )
        overlay = overlay_heatmap(img_array, heatmap)

        pred_label = label_encoder.inverse_transform([pred_idx])[0]
        confidence = float(np.max(pred_probs))

        axes[i, 0].imshow(img_array)
        axes[i, 0].set_title(f"Original\nTrue: {row['true_label']}", fontsize=10)
        axes[i, 0].axis("off")

        axes[i, 1].imshow(heatmap, cmap="jet")
        axes[i, 1].set_title("Grad-CAM Heatmap", fontsize=10)
        axes[i, 1].axis("off")

        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title(f"Overlay\nPred: {pred_label} ({confidence:.3f})", fontsize=10)
        axes[i, 2].axis("off")

    plt.suptitle("Grad-CAM Explanations for Transfer Learning Model", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig("gradcam_examples.png", dpi=150, bbox_inches="tight")
    plt.show()

print("Grad-CAM generation complete. Saved figure: gradcam_examples.png")

In [ ]:
# get all the visualization stuff and put it in the report lol